# 02 - Sanity Checks

Quick checks that the `gpt2_chatbot` package works end to end:

1. Model builds and a forward pass returns logits of the expected shape.
2. Parameter count is in the right ballpark for GPT-2 small (~124M / ~163M with untied head).
3. The causal mask prevents attending to future tokens.
4. Tokenizer round-trips text.
5. Greedy and sampled generation both run.
6. (Optional) Load official OpenAI GPT-2 weights and generate coherent text.

This notebook only reads from the package; it does not modify any source files.

In [1]:
import sys
from pathlib import Path

# Make the src-layout package importable when running from notebooks/.
SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import torch

from gpt2_chatbot.model import GPTModel, get_config
from gpt2_chatbot.tokenizer import get_tokenizer, text_to_token_ids, token_ids_to_text
from gpt2_chatbot.inference import generate, generate_text_simple

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

torch 2.6.0 | device: cpu


## 1. Build model + forward-pass shape check

In [2]:
cfg = get_config("gpt2-small (124M)", context_length=256)
model = GPTModel(cfg).to(device).eval()

batch, seq_len = 2, 8
dummy = torch.randint(0, cfg["vocab_size"], (batch, seq_len), device=device)
logits = model(dummy)

expected = (batch, seq_len, cfg["vocab_size"])
assert logits.shape == expected, f"got {tuple(logits.shape)}, expected {expected}"
print("OK - logits shape:", tuple(logits.shape))

OK - logits shape: (2, 8, 50257)


## 2. Parameter count

GPT-2 small has ~124M parameters with the token embedding tied to the output head. Here the head is a separate `nn.Linear`, so the raw total is higher (~163M). We report both.

In [3]:
total = sum(p.numel() for p in model.parameters())
tied = total - model.out_head.weight.numel()  # subtract the untied output head
print(f"Total params:            {total:,}")
print(f"With tied embeddings:    {tied:,}")
assert 120_000_000 < tied < 130_000_000, "tied param count outside expected GPT-2 small range"
print("OK - parameter count in expected range")

Total params:            162,419,712
With tied embeddings:    123,822,336
OK - parameter count in expected range


## 3. Causal mask check

Changing a token should only affect logits at that position and later ones, never earlier positions.

In [4]:
seq_len = 6
ids = torch.randint(0, cfg["vocab_size"], (1, seq_len), device=device)
with torch.no_grad():
    base = model(ids)

# Flip the token at the last position.
ids_mod = ids.clone()
ids_mod[0, -1] = (ids_mod[0, -1] + 1) % cfg["vocab_size"]
with torch.no_grad():
    changed = model(ids_mod)

earlier_diff = (base[:, :-1, :] - changed[:, :-1, :]).abs().max().item()
last_diff = (base[:, -1, :] - changed[:, -1, :]).abs().max().item()
print(f"max change at earlier positions: {earlier_diff:.3e} (should be ~0)")
print(f"max change at last position:     {last_diff:.3e} (should be > 0)")
assert earlier_diff < 1e-5, "future token leaked into earlier positions - causal mask broken"
assert last_diff > 0, "changing the last token had no effect"
print("OK - attention is causal")

max change at earlier positions: 0.000e+00 (should be ~0)
max change at last position:     2.748e+00 (should be > 0)
OK - attention is causal


## 4. Tokenizer round-trip

In [5]:
tok = get_tokenizer()
text = "Every effort moves you"
ids = text_to_token_ids(text, tok)
roundtrip = token_ids_to_text(ids, tok)
print("ids:", ids.tolist())
print("roundtrip:", repr(roundtrip))
assert roundtrip == text, "tokenizer did not round-trip"
print("OK - tokenizer round-trips")

ids: [[6109, 3626, 6100, 345]]
roundtrip: 'Every effort moves you'
OK - tokenizer round-trips


## 5. Generation runs (random weights - output will be gibberish)

In [6]:
start = text_to_token_ids("Every effort moves you", tok).to(device)

greedy = generate_text_simple(model, start, max_new_tokens=10, context_size=cfg["context_length"])
sampled = generate(model, start, max_new_tokens=10, context_size=cfg["context_length"],
                   top_k=25, temperature=1.4)

print("greedy :", repr(token_ids_to_text(greedy, tok)))
print("sampled:", repr(token_ids_to_text(sampled, tok)))
assert greedy.shape[1] == start.shape[1] + 10
assert sampled.shape[1] <= start.shape[1] + 10
print("OK - both generation paths run")

greedy : 'Every effort moves you rentingetic wasnم refres RexMeCHicular stren'
sampled: 'Every effort moves you Laur RebellRR Aman AjLab impression Meadolk Premier'
OK - both generation paths run


## 6. (Optional) Load pretrained GPT-2 weights

This requires the `gpt_download` helper and `tensorflow` to fetch the official OpenAI checkpoints. If they aren't available, the cell skips gracefully. With real weights, generation should produce coherent English.

In [7]:
try:
    from gpt2_chatbot.model import load_hf_weights_into_gpt

    pretrained_cfg = get_config("gpt2-small (124M)", context_length=1024, qkv_bias=True)
    gpt = GPTModel(pretrained_cfg)
    load_hf_weights_into_gpt(gpt, model_name="gpt2")  # downloads ~500MB the first time
    gpt.to(device).eval()

    out = generate(
        gpt,
        text_to_token_ids("Every effort moves you", tok).to(device),
        max_new_tokens=25,
        context_size=pretrained_cfg["context_length"],
        top_k=50,
        temperature=1.0,
    )
    print(token_ids_to_text(out, tok))
except Exception as e:
    print("Skipping pretrained-weight check:", type(e).__name__, e)

Skipping pretrained-weight check: ModuleNotFoundError No module named 'gpt_download'


## 7. Chat format demo (Step 1)

Play with the chat template here. Edit `conv` below and re-run to see:
- the `(text, trainable)` segments the format is built from,
- the training string vs the inference prompt,
- the token ids and the per-token assistant mask (which tokens the SFT loss
  will actually be computed on).

In [8]:
from gpt2_chatbot.data import chat_format as cf
from gpt2_chatbot.data import render_for_training, render_for_inference, encode_conversation

# >>> Edit this conversation and re-run the cells below. <<<
conv = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "4"},
]

print("Segments (text, trainable):")
for text, trainable in cf._segments(conv, add_generation_prompt=False):
    flag = "TRAIN" if trainable else "     "
    print(f"  [{flag}] {text!r}")

Segments (text, trainable):
  [     ] '<|user|>\nWhat is 2+2?\n'
  [     ] '<|assistant|>\n'
  [TRAIN] '4<|endoftext|>\n'


In [9]:
print("=== TRAINING STRING (full, ends with stop token) ===")
print(render_for_training(conv))
print("=== INFERENCE PROMPT (drop last assistant turn; model generates the rest) ===")
print(render_for_inference(conv[:-1]))

=== TRAINING STRING (full, ends with stop token) ===
<|user|>
What is 2+2?
<|assistant|>
4<|endoftext|>

=== INFERENCE PROMPT (drop last assistant turn; model generates the rest) ===
<|user|>
What is 2+2?
<|assistant|>



In [10]:
ids, mask = encode_conversation(conv, tok)
print(f"{len(ids)} tokens total, {sum(mask)} trainable (assistant) tokens\n")
print(f"{'idx':>3}  {'id':>6}  {'trained':>7}  token")
print('-' * 42)
for i, (tid, m) in enumerate(zip(ids, mask)):
    piece = tok.decode([tid]).replace('\n', '\\n')
    print(f"{i:>3}  {tid:>6}  {str(m):>7}  {piece!r}")

23 tokens total, 3 trainable (assistant) tokens

idx      id  trained  token
------------------------------------------
  0      27    False  '<'
  1      91    False  '|'
  2    7220    False  'user'
  3      91    False  '|'
  4      29    False  '>'
  5     198    False  '\\n'
  6    2061    False  'What'
  7     318    False  ' is'
  8     362    False  ' 2'
  9      10    False  '+'
 10      17    False  '2'
 11      30    False  '?'
 12     198    False  '\\n'
 13      27    False  '<'
 14      91    False  '|'
 15     562    False  'ass'
 16   10167    False  'istant'
 17      91    False  '|'
 18      29    False  '>'
 19     198    False  '\\n'
 20      19     True  '4'
 21   50256     True  '<|endoftext|>'
 22     198     True  '\\n'


In [11]:
# The trained span should decode to exactly the assistant reply + stop token.
trained_ids = [i for i, m in zip(ids, mask) if m]
print("Trained tokens decode to:", repr(tok.decode(trained_ids)))
print("Stop token id (<|endoftext|>):", cf.EOT, "->", tok.encode(cf.EOT, allowed_special={cf.EOT}))

Trained tokens decode to: '4<|endoftext|>\n'
Stop token id (<|endoftext|>): <|endoftext|> -> [50256]
